[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-1/router.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239412-lesson-5-router)

# Router

## Review

We built a graph that uses `messages` as state and a chat model with bound tools.

We saw that the graph can:

* Return a tool call
* Return a natural language response

## Goals

We can think of this as a router, where the chat model routes between a direct response or a tool call based upon the user input.

This is a simple example of an agent, where the LLM is directing the control flow either by calling a tool or just responding directly. 

![Screenshot 2024-08-21 at 9.24.09 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbac6543c3d4df239a4ed1_router1.png)

Let's extend our graph to work with either output! 

For this, we can use two ideas:

(1) Add a node that will call our tool.

(2) Add a conditional edge that will look at the chat model output, and route to our tool calling node or simply end if no tool call is performed. 



In [1]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langgraph langgraph-prebuilt

In [3]:
import sys
from pathlib import Path

# Add parent directory to path (only once at top of notebook)
sys.path.insert(0, str(Path.cwd().parent))

# Now import normally
from utils import (
    setup_environment,
    load_env_file,
    _set_env
)

# Use the functions
setup_environment(
    ["LANGSMITH_API_KEY", "GROQ_API_KEY", "TAVILY_API_KEY"],
    env_file="../.env"
)

🚀 Setting up environment variables...
Loading environment variables from ../.env
✅ Environment variables loaded from .env file

📋 Checking required environment variables:
✅ LANGSMITH_API_KEY already configured
✅ GROQ_API_KEY already configured
✅ TAVILY_API_KEY already configured

🎉 Environment setup complete!


In [4]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

In [5]:
# Free Groq models instead of OpenAI
from langchain_groq import ChatGroq

llm1 = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
llm2 = ChatGroq(model="qwen/qwen3-32b", temperature=0)

llm = llm1
llm_with_tools = llm.bind_tools([multiply])

 We use the [built-in `ToolNode`](https://langchain-ai.github.io/langgraph/reference/prebuilt/?h=tools+condition#toolnode) and simply pass a list of our tools to initialize it. 
 
 We use the [built-in `tools_condition`](https://langchain-ai.github.io/langgraph/reference/prebuilt/?h=tools+condition#tools_condition) as our conditional edge.

In [6]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

# Node
def tool_calling_llm(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

# Build graph
builder = StateGraph(MessagesState)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode([multiply]))
builder.add_edge(START, "tool_calling_llm")
# builder.add_conditional_edges(
#     "x_tool_calling_llm",
#     # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
#     # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
#     tools_condition,
# )

builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condition,
    {
        "tools": "tools",  # Route to tools if tool call
        END: END             # Route to end if no tool call
    }
)


builder.add_edge("tools", END)
graph = builder.compile()

In [7]:
from langchain_core.runnables.graph_mermaid import MermaidDrawMethod
from IPython.display import Image, display

# # View
# display(Image(graph.get_graph().draw_mermaid_png()))

# # ASCII visualization (always works)
# print("Graph Structure:")
# print(graph.get_graph().draw_ascii())

try:
    # Try local rendering
    display(Image(graph.get_graph().draw_mermaid_png(
        draw_method=MermaidDrawMethod.PYPPETEER
    )))
except Exception as e:
    print(f"Pyppeteer failed: {e}")
    # Fallback to ASCII
    print(graph.get_graph().draw_ascii())

Pyppeteer failed: asyncio.run() cannot be called from a running event loop
        +-----------+     
        | __start__ |     
        +-----------+     
              *           
              *           
              *           
    +------------------+  
    | tool_calling_llm |  
    +------------------+  
         ..        ..     
       ..            .    
      .               ..  
+-------+               . 
| tools |             ..  
+-------+            .    
         **        ..     
           **    ..       
             *  .         
         +---------+      
         | __end__ |      
         +---------+      


/var/folders/tr/_8gn26jn35x_vmmf6p7s4tlr0000gn/T/ipykernel_98373/253371735.py:19: RuntimeWarning: coroutine '_render_mermaid_using_pyppeteer' was never awaited
  print(graph.get_graph().draw_ascii())


In [8]:
from langchain_core.messages import HumanMessage
messages = [
    HumanMessage(content="Hello, what is 2 multiplied by 2?"),
    HumanMessage(content="Hello, what is 3 multiplied by 2?"),
    HumanMessage(content="Hello, what is 2 multiplied by 10?"),
    HumanMessage(content="Hello, what is 20 multiplied by 10?"),

]
messages = graph.invoke({"messages": messages})
for m in messages['messages']:
    m.pretty_print()

================================ Human Message =================================

Hello, what is 2 multiplied by 2?
================================ Human Message =================================

Hello, what is 3 multiplied by 2?
================================ Human Message =================================

Hello, what is 2 multiplied by 10?
================================ Human Message =================================

Hello, what is 20 multiplied by 10?
================================== Ai Message ==================================
Tool Calls:
  multiply (ncr40fy7d)
 Call ID: ncr40fy7d
  Args:
    a: 20
    b: 10
================================= Tool Message =================================
Name: multiply

200


In [9]:
messages

{'messages': [HumanMessage(content='Hello, what is 2 multiplied by 2?', additional_kwargs={}, response_metadata={}, id='f54f384e-06aa-4d33-aa62-c1c24fb59632'),
  HumanMessage(content='Hello, what is 3 multiplied by 2?', additional_kwargs={}, response_metadata={}, id='e9b1ab1e-3ec8-4205-92e6-6ffbce9329ed'),
  HumanMessage(content='Hello, what is 2 multiplied by 10?', additional_kwargs={}, response_metadata={}, id='43599b1b-e088-4310-9049-9a9511ecd1d6'),
  HumanMessage(content='Hello, what is 20 multiplied by 10?', additional_kwargs={}, response_metadata={}, id='0587b9e1-5017-4db7-b5dc-d02ef736d568'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ncr40fy7d', 'function': {'arguments': '{"a":20,"b":10}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 306, 'total_tokens': 325, 'completion_time': 0.04407481, 'prompt_time': 0.025902321, 'queue_time': 0.084620782, 'total_time': 0.069977131}, 'model_na

Now, we can see that the graph runs the tool!

It responds with a `ToolMessage`. 

## LangGraph Studio

**⚠️ DISCLAIMER**

Since the filming of these videos, we've updated Studio so that it can be run locally and opened in your browser. This is now the preferred way to run Studio (rather than using the Desktop App as shown in the video). See documentation [here](https://langchain-ai.github.io/langgraph/concepts/langgraph_studio/#local-development-server) on the local development server and [here](https://langchain-ai.github.io/langgraph/how-tos/local-studio/#run-the-development-server). To start the local development server, run the following command in your terminal in the `/studio` directory in this module:

```
langgraph dev
```

You should see the following output:
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

Open your browser and navigate to the Studio UI: `https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024`.
Load the `router` in Studio, which uses `module-1/studio/router.py` set in `module-1/studio/langgraph.json`.

In [10]:
if 'google.colab' in str(get_ipython()):
    raise Exception("Unfortunately LangGraph Studio is currently not supported on Google Colab")